# 29_PF004_PF007_final_models_registry_metrics

## PF-004 a PF-007 — Modelos finales, configuración única y métricas

Este notebook consolida los artefactos finales de modelos para el producto final.

No entrena por defecto. Toma los checkpoints validados de E10 y E12, los copia a `models/final/`, registra hashes, genera `model_registry_final.json`, métricas finales y documentación de cierre.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, json, hashlib, shutil, datetime
import pandas as pd
import numpy as np

PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")
REPO_ROOT = PFI_ROOT / "repo"
RESULTS_ROOT = PFI_ROOT / "results" / "PF004_PF007_final_models"
MODELS_ROOT = PFI_ROOT / "models"
FINAL_MODELS_ROOT = MODELS_ROOT / "final"
DOCS_ROOT = PFI_ROOT / "docs"

for p in [RESULTS_ROOT, FINAL_MODELS_ROOT, DOCS_ROOT, REPO_ROOT / "config", REPO_ROOT / "docs", REPO_ROOT / "backlogProducto"]:
    p.mkdir(parents=True, exist_ok=True)

print("PFI_ROOT:", PFI_ROOT)
print("REPO_ROOT:", REPO_ROOT, "| exists:", REPO_ROOT.exists())
print("MODELS_ROOT:", MODELS_ROOT, "| exists:", MODELS_ROOT.exists())
print("FINAL_MODELS_ROOT:", FINAL_MODELS_ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)


Mounted at /content/drive
PFI_ROOT: /content/drive/MyDrive/PFI_MVP
REPO_ROOT: /content/drive/MyDrive/PFI_MVP/repo | exists: True
MODELS_ROOT: /content/drive/MyDrive/PFI_MVP/models | exists: True
FINAL_MODELS_ROOT: /content/drive/MyDrive/PFI_MVP/models/final
RESULTS_ROOT: /content/drive/MyDrive/PFI_MVP/results/PF004_PF007_final_models


## 1. Entradas esperadas

In [2]:
DATA_FREEZE_CONFIG = REPO_ROOT / "config" / "data_freeze_config.json"

AXIAL_SRC_CANDIDATES = [
    MODELS_ROOT / "E10_axial_t2_final_training_clean_best.pt",
    PFI_ROOT / "results" / "E10_axial_t2_final_training_clean" / "E10_axial_t2_final_training_clean_best.pt",
]

SAGITTAL_SRC_CANDIDATES = [
    MODELS_ROOT / "E12_sagittal_multiclass_final_best.pt",
    PFI_ROOT / "results" / "E12_sagittal_final_training_clean" / "E12_sagittal_multiclass_final_best.pt",
    PFI_ROOT / "results" / "E5_multiclase_agrupado" / "E5_multiclass_unet2d_grouped_best.pt",
]

def first_existing(candidates):
    for p in candidates:
        if p.exists():
            return p
    return None

axial_src = first_existing(AXIAL_SRC_CANDIDATES)
sagittal_src = first_existing(SAGITTAL_SRC_CANDIDATES)

print("DATA_FREEZE_CONFIG:", DATA_FREEZE_CONFIG, "->", DATA_FREEZE_CONFIG.exists())
print("axial_src:", axial_src, "->", axial_src.exists() if axial_src else False)
print("sagittal_src:", sagittal_src, "->", sagittal_src.exists() if sagittal_src else False)

data_freeze = json.loads(DATA_FREEZE_CONFIG.read_text(encoding="utf-8")) if DATA_FREEZE_CONFIG.exists() else {}
print("data_freeze keys:", list(data_freeze.keys())[:10])


DATA_FREEZE_CONFIG: /content/drive/MyDrive/PFI_MVP/repo/config/data_freeze_config.json -> True
axial_src: /content/drive/MyDrive/PFI_MVP/models/E10_axial_t2_final_training_clean_best.pt -> True
sagittal_src: /content/drive/MyDrive/PFI_MVP/models/E12_sagittal_multiclass_final_best.pt -> True
data_freeze keys: ['freeze_name', 'freeze_timestamp', 'pfi_root', 'repo_root', 'data_root', 'results_root', 'docs_root', 'models_root', 'datasets', 'final_models']


## 2. Copia de artefactos finales y hashes

In [3]:
def sha256_file(path: Path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

final_artifacts = []
model_specs = [
    {
        "model_key": "axial_t2_alkafri",
        "plane": "axial",
        "src": axial_src,
        "dst": FINAL_MODELS_ROOT / "axial_t2_alkafri_final_best.pt",
        "source_stage": "E10",
        "dataset_key": "alkafri_sudirman_axial",
        "status": "final_checkpoint_consolidated",
    },
    {
        "model_key": "sagittal_spider",
        "plane": "sagittal",
        "src": sagittal_src,
        "dst": FINAL_MODELS_ROOT / "sagittal_spider_multiclass_final_best.pt",
        "source_stage": "E12",
        "dataset_key": "spider_sagittal",
        "status": "final_checkpoint_consolidated",
    },
]

for spec in model_specs:
    src = spec["src"]
    dst = spec["dst"]
    exists = src is not None and src.exists()
    if exists:
        shutil.copy2(src, dst)
        row = {
            **{k: v for k, v in spec.items() if k not in ["src", "dst"]},
            "source_path": str(src),
            "final_path": str(dst),
            "exists": True,
            "size_bytes": dst.stat().st_size,
            "sha256": sha256_file(dst),
        }
    else:
        row = {
            **{k: v for k, v in spec.items() if k not in ["src", "dst"]},
            "source_path": str(src) if src else None,
            "final_path": str(dst),
            "exists": False,
            "size_bytes": None,
            "sha256": None,
        }
    final_artifacts.append(row)

artifacts_df = pd.DataFrame(final_artifacts)
artifacts_csv = RESULTS_ROOT / "PF004_PF007_model_artifacts.csv"
artifacts_df.to_csv(artifacts_csv, index=False)

display(artifacts_df)
print("Artifacts CSV:", artifacts_csv)


,model_key,plane,source_stage,dataset_key,status,source_path,final_path,exists,size_bytes,sha256
0,axial_t2_alkafri,axial,E10,alkafri_sudirman_axial,final_checkpoint_consolidated,/content/drive/MyDrive/PFI_MVP/models/E10_axia...,/content/drive/MyDrive/PFI_MVP/models/final/ax...,True,1973243,a8b91f563e101c9c4a3cf2c0ec84af29cbcfabd76eb76b...
1,sagittal_spider,sagittal,E12,spider_sagittal,final_checkpoint_consolidated,/content/drive/MyDrive/PFI_MVP/models/E12_sagi...,/content/drive/MyDrive/PFI_MVP/models/final/sa...,True,1975947,7dd393cc750311c98003516d8110136310c31e8b6f0f00...


Artifacts CSV: /content/drive/MyDrive/PFI_MVP/results/PF004_PF007_final_models/PF004_PF007_model_artifacts.csv


## 3. Métricas finales por modelo

In [4]:
metrics_rows = [
    {"model_key":"axial_t2_alkafri","source_stage":"E10","dataset_key":"alkafri_sudirman_axial","plane":"axial","split":"val","metric_scope":"macro_no_background_including_raw0","dice":0.7054,"iou":0.6364,"notes":"Incluye raw_0; reportar junto con métrica útil."},
    {"model_key":"axial_t2_alkafri","source_stage":"E10","dataset_key":"alkafri_sudirman_axial","plane":"axial","split":"val","metric_scope":"useful_classes_excluding_raw0","dice":0.8817,"iou":0.7955,"notes":"Métrica útil para clases axiales estables."},
    {"model_key":"axial_t2_alkafri","source_stage":"E10","dataset_key":"alkafri_sudirman_axial","plane":"axial","split":"test","metric_scope":"macro_no_background_including_raw0","dice":0.6587,"iou":0.5628,"notes":"Incluye raw_0; reportar junto con métrica útil."},
    {"model_key":"axial_t2_alkafri","source_stage":"E10","dataset_key":"alkafri_sudirman_axial","plane":"axial","split":"test","metric_scope":"useful_classes_excluding_raw0","dice":0.8167,"iou":0.7001,"notes":"Métrica principal recomendada para comunicación del módulo axial."},
    {"model_key":"sagittal_spider","source_stage":"E12/E5","dataset_key":"spider_sagittal","plane":"sagittal","split":"val","metric_scope":"macro_no_background","dice":0.8392,"iou":0.7466,"notes":"Validación multiclase agrupada."},
    {"model_key":"sagittal_spider","source_stage":"E12/E5","dataset_key":"spider_sagittal","plane":"sagittal","split":"holdout/test","metric_scope":"macro_no_background","dice":0.8311,"iou":0.7309,"notes":"Holdout multiclase agrupado."},
]

class_metrics_rows = [
    {"model_key":"axial_t2_alkafri","split":"test","class_name":"background","dice":0.99296,"notes":"fondo"},
    {"model_key":"axial_t2_alkafri","split":"test","class_name":"raw_0","dice":0.0264,"notes":"clase minoritaria/intermitente, analizada en E11"},
    {"model_key":"axial_t2_alkafri","split":"test","class_name":"raw_50","dice":0.9348,"notes":"clase útil axial"},
    {"model_key":"axial_t2_alkafri","split":"test","class_name":"raw_100","dice":0.8485,"notes":"clase útil axial"},
    {"model_key":"axial_t2_alkafri","split":"test","class_name":"raw_150","dice":0.7997,"notes":"clase útil axial"},
    {"model_key":"axial_t2_alkafri","split":"test","class_name":"raw_200","dice":0.6840,"notes":"clase útil axial"},
    {"model_key":"sagittal_spider","split":"holdout/test","class_name":"background","dice":0.9745,"notes":"fondo"},
    {"model_key":"sagittal_spider","split":"holdout/test","class_name":"vertebra_group","dice":0.8516,"notes":"clase sagital útil"},
    {"model_key":"sagittal_spider","split":"holdout/test","class_name":"canal","dice":0.8645,"notes":"clase sagital útil"},
    {"model_key":"sagittal_spider","split":"holdout/test","class_name":"disc_group","dice":0.7978,"notes":"clase sagital útil"},
]

metrics_df = pd.DataFrame(metrics_rows)
class_metrics_df = pd.DataFrame(class_metrics_rows)

metrics_csv = RESULTS_ROOT / "PF007_final_model_metrics.csv"
class_metrics_csv = RESULTS_ROOT / "PF007_final_model_metrics_by_class.csv"
metrics_df.to_csv(metrics_csv, index=False)
class_metrics_df.to_csv(class_metrics_csv, index=False)

display(metrics_df)
display(class_metrics_df)

print("Metrics:", metrics_csv)
print("Class metrics:", class_metrics_csv)


,model_key,source_stage,dataset_key,plane,split,metric_scope,dice,iou,notes
0,axial_t2_alkafri,E10,alkafri_sudirman_axial,axial,val,macro_no_background_including_raw0,0.7054,0.6364,Incluye raw_0; reportar junto con métrica útil.
1,axial_t2_alkafri,E10,alkafri_sudirman_axial,axial,val,useful_classes_excluding_raw0,0.8817,0.7955,Métrica útil para clases axiales estables.
2,axial_t2_alkafri,E10,alkafri_sudirman_axial,axial,test,macro_no_background_including_raw0,0.6587,0.5628,Incluye raw_0; reportar junto con métrica útil.
3,axial_t2_alkafri,E10,alkafri_sudirman_axial,axial,test,useful_classes_excluding_raw0,0.8167,0.7001,Métrica principal recomendada para comunicació...
4,sagittal_spider,E12/E5,spider_sagittal,sagittal,val,macro_no_background,0.8392,0.7466,Validación multiclase agrupada.
5,sagittal_spider,E12/E5,spider_sagittal,sagittal,holdout/test,macro_no_background,0.8311,0.7309,Holdout multiclase agrupado.


,model_key,split,class_name,dice,notes
0,axial_t2_alkafri,test,background,0.99296,fondo
1,axial_t2_alkafri,test,raw_0,0.02640,"clase minoritaria/intermitente, analizada en E11"
2,axial_t2_alkafri,test,raw_50,0.93480,clase útil axial
3,axial_t2_alkafri,test,raw_100,0.84850,clase útil axial
4,axial_t2_alkafri,test,raw_150,0.79970,clase útil axial
5,axial_t2_alkafri,test,raw_200,0.68400,clase útil axial
6,sagittal_spider,holdout/test,background,0.97450,fondo
7,sagittal_spider,holdout/test,vertebra_group,0.85160,clase sagital útil
8,sagittal_spider,holdout/test,canal,0.86450,clase sagital útil
9,sagittal_spider,holdout/test,disc_group,0.79780,clase sagital útil


Metrics: /content/drive/MyDrive/PFI_MVP/results/PF004_PF007_final_models/PF007_final_model_metrics.csv
Class metrics: /content/drive/MyDrive/PFI_MVP/results/PF004_PF007_final_models/PF007_final_model_metrics_by_class.csv


## 4. Model registry final

In [5]:
def artifact_by_key(key):
    row = artifacts_df[artifacts_df["model_key"] == key]
    if len(row) == 0:
        return {}
    return row.iloc[0].to_dict()

registry = {
    "version": "PF004_PF007_final_registry_v1",
    "created_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "data_freeze_config": str(DATA_FREEZE_CONFIG),
    "human_review_required": True,
    "not_clinical_diagnosis": True,
    "not_real_3d_reconstruction": True,
    "models": {
        "axial_t2_alkafri": {
            "plane": "axial",
            "dataset_key": "alkafri_sudirman_axial",
            "source_stage": "E10",
            "architecture": "U-Net 2D multiclass",
            "checkpoint_path": artifact_by_key("axial_t2_alkafri").get("final_path"),
            "sha256": artifact_by_key("axial_t2_alkafri").get("sha256"),
            "num_classes": 6,
            "class_names": {"0":"background_250","1":"raw_0","2":"raw_50","3":"raw_100","4":"raw_150","5":"raw_200"},
            "primary_metric": {"split":"test","name":"dice_useful_classes_excluding_raw0","value":0.8167},
            "secondary_metric": {"split":"test","name":"dice_macro_no_background_including_raw0","value":0.6587},
            "notes": "raw_0 se reporta separado por ser clase minoritaria/intermitente documentada en E11.",
        },
        "sagittal_spider": {
            "plane": "sagittal",
            "dataset_key": "spider_sagittal",
            "source_stage": "E12",
            "architecture": "U-Net 2D multiclass grouped",
            "checkpoint_path": artifact_by_key("sagittal_spider").get("final_path"),
            "sha256": artifact_by_key("sagittal_spider").get("sha256"),
            "num_classes": 4,
            "class_names": {"0":"background","1":"vertebra_group","2":"canal","3":"disc_group"},
            "primary_metric": {"split":"holdout/test","name":"dice_macro_no_background","value":0.8311},
            "secondary_metric": {"split":"val","name":"dice_macro_no_background","value":0.8392},
            "notes": "Modelo sagital consolidado en E12 sin reentrenamiento adicional.",
        },
    },
}

registry_results_path = RESULTS_ROOT / "PF006_model_registry_final.json"
registry_repo_path = REPO_ROOT / "config" / "model_registry_final.json"

registry_results_path.write_text(json.dumps(registry, indent=2, ensure_ascii=False), encoding="utf-8")
registry_repo_path.write_text(json.dumps(registry, indent=2, ensure_ascii=False), encoding="utf-8")

print("Registry results:", registry_results_path)
print("Registry repo:", registry_repo_path)


Registry results: /content/drive/MyDrive/PFI_MVP/results/PF004_PF007_final_models/PF006_model_registry_final.json
Registry repo: /content/drive/MyDrive/PFI_MVP/repo/config/model_registry_final.json


## 5. Documentación de modelos finales

In [ ]:
doc_lines = []
doc_lines.append("# PF-004 a PF-007 - Modelos finales y métricas")
doc_lines.append("")
doc_lines.append("## Objetivo")
doc_lines.append("Consolidar checkpoints finales, configurar una fuente única de modelos y documentar métricas finales para el producto.")
doc_lines.append("")
doc_lines.append("## Artefactos finales")
doc_lines.append("")
doc_lines.append("| Modelo | Plano | Dataset | Checkpoint final | SHA256 |")
doc_lines.append("|---|---|---|---|---|")
for _, row in artifacts_df.iterrows():
    doc_lines.append(f"| {row['model_key']} | {row['plane']} | {row['dataset_key']} | `{row['final_path']}` | `{row['sha256']}` |")
doc_lines.append("")
doc_lines.append("## Métricas principales")
doc_lines.append("")
doc_lines.append("| Modelo | Split | Alcance | Dice | IoU |")
doc_lines.append("|---|---|---|---:|---:|")
for _, row in metrics_df.iterrows():
    doc_lines.append(f"| {row['model_key']} | {row['split']} | {row['metric_scope']} | {row['dice']:.4f} | {row['iou']:.4f} |")
doc_lines.append("")
doc_lines.append("## Decisiones metodológicas")
doc_lines.append("")
doc_lines.append("- El modelo sagital se consolida desde E12; no se reentrena si el checkpoint ya está validado.")
doc_lines.append("- El modelo axial se consolida desde E10; `raw_0` se reporta por separado por su comportamiento minoritario/intermitente documentado en E11.")
doc_lines.append("- Las métricas finales no incluyen SSMSpine ni datasets opcionales.")
doc_lines.append("- Los modelos se presentan como soporte técnico para segmentación asistida, no como diagnóstico clínico.")
doc_lines.append("- Toda salida del sistema requiere revisión profesional.")
doc_lines.append("")
doc_lines.append("## Uso posterior")
doc_lines.append("El archivo `config/model_registry_final.json` será usado por el servicio Python para cargar el modelo correcto según plano y para exponer metadatos de modelo al backend.")

doc_md = "\n".join(doc_lines)

docs_result_path = DOCS_ROOT / "PF004_PF007_final_models_metrics.md"
docs_repo_path = REPO_ROOT / "docs" / "PF004_PF007_final_models_metrics.md"
docs_result_path.write_text(doc_md, encoding="utf-8")
docs_repo_path.write_text(doc_md, encoding="utf-8")

print("Docs result:", docs_result_path)
print("Docs repo:", docs_repo_path)


## 6. Checks finales

In [7]:
checks = []
def add_check(name, ok, detail=""):
    checks.append({"check": name, "ok": bool(ok), "detail": detail})

add_check("data_freeze_config_exists", DATA_FREEZE_CONFIG.exists(), str(DATA_FREEZE_CONFIG))
add_check("axial_final_checkpoint_exists", (FINAL_MODELS_ROOT / "axial_t2_alkafri_final_best.pt").exists(), str(FINAL_MODELS_ROOT / "axial_t2_alkafri_final_best.pt"))
add_check("sagittal_final_checkpoint_exists", (FINAL_MODELS_ROOT / "sagittal_spider_multiclass_final_best.pt").exists(), str(FINAL_MODELS_ROOT / "sagittal_spider_multiclass_final_best.pt"))
add_check("artifacts_csv_written", artifacts_csv.exists(), str(artifacts_csv))
add_check("metrics_csv_written", metrics_csv.exists(), str(metrics_csv))
add_check("class_metrics_csv_written", class_metrics_csv.exists(), str(class_metrics_csv))
add_check("registry_results_written", registry_results_path.exists(), str(registry_results_path))
add_check("registry_repo_written", registry_repo_path.exists(), str(registry_repo_path))
add_check("docs_repo_written", docs_repo_path.exists(), str(docs_repo_path))
add_check("all_artifacts_have_sha256", artifacts_df["sha256"].notna().all(), "sha256 not null for final checkpoints")
add_check("only_final_scope_datasets_in_metrics", set(metrics_df["dataset_key"]).issubset({"spider_sagittal","alkafri_sudirman_axial"}), str(sorted(metrics_df["dataset_key"].unique())))

checks_df = pd.DataFrame(checks)
checks_csv = RESULTS_ROOT / "PF004_PF007_checks.csv"
checks_df.to_csv(checks_csv, index=False)
display(checks_df)

all_ok = bool(checks_df["ok"].all())
print("Checks:", checks_csv)
print("All OK:", all_ok)


,check,ok,detail
0,data_freeze_config_exists,True,/content/drive/MyDrive/PFI_MVP/repo/config/dat...
1,axial_final_checkpoint_exists,True,/content/drive/MyDrive/PFI_MVP/models/final/ax...
2,sagittal_final_checkpoint_exists,True,/content/drive/MyDrive/PFI_MVP/models/final/sa...
3,artifacts_csv_written,True,/content/drive/MyDrive/PFI_MVP/results/PF004_P...
4,metrics_csv_written,True,/content/drive/MyDrive/PFI_MVP/results/PF004_P...
5,class_metrics_csv_written,True,/content/drive/MyDrive/PFI_MVP/results/PF004_P...
6,registry_results_written,True,/content/drive/MyDrive/PFI_MVP/results/PF004_P...
7,registry_repo_written,True,/content/drive/MyDrive/PFI_MVP/repo/config/mod...
8,docs_repo_written,True,/content/drive/MyDrive/PFI_MVP/repo/docs/PF004...
9,all_artifacts_have_sha256,True,sha256 not null for final checkpoints


Checks: /content/drive/MyDrive/PFI_MVP/results/PF004_PF007_final_models/PF004_PF007_checks.csv
All OK: True


## 7. Reporte final y cierre de backlog

In [8]:
report = {
    "notebook": "29_PF004_PF007_final_models_registry_metrics",
    "goal": "consolidate final model artifacts, final model registry and final metrics",
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    "inputs": {
        "data_freeze_config": str(DATA_FREEZE_CONFIG),
        "axial_source_checkpoint": str(axial_src) if axial_src else None,
        "sagittal_source_checkpoint": str(sagittal_src) if sagittal_src else None,
    },
    "outputs": {
        "final_models_root": str(FINAL_MODELS_ROOT),
        "model_artifacts_csv": str(artifacts_csv),
        "model_registry_results": str(registry_results_path),
        "model_registry_repo": str(registry_repo_path),
        "final_model_metrics_csv": str(metrics_csv),
        "final_model_metrics_by_class_csv": str(class_metrics_csv),
        "checks_csv": str(checks_csv),
        "docs_repo": str(docs_repo_path),
    },
    "summary": {
        "models_consolidated": int(artifacts_df["exists"].sum()),
        "model_keys": artifacts_df["model_key"].tolist(),
        "all_checks_ok": all_ok,
        "primary_metrics": {
            "axial_t2_alkafri": {
                "test_dice_useful_classes_excluding_raw0": 0.8167,
                "test_dice_macro_no_background_including_raw0": 0.6587,
            },
            "sagittal_spider": {
                "holdout_test_dice_macro_no_background": 0.8311,
                "val_dice_macro_no_background": 0.8392,
            },
        },
    },
    "methodological_policy": {
        "human_review_required": True,
        "not_clinical_diagnosis": True,
        "not_real_3d_reconstruction": True,
        "optional_datasets_excluded_from_final_metrics": True,
    },
    "decision": "PF004_PF007_final_models_ready_for_product_inference" if all_ok else "PF004_PF007_requires_fix",
}

report_path = RESULTS_ROOT / "PF004_PF007_report.json"
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")

closure_md = f"""# Cierre PF-004 a PF-007 - Modelos finales y métricas

Estado: {"cerrado" if all_ok else "requiere corrección"}.

## Decisión

```text
{report["decision"]}
```

## Modelos consolidados

- axial_t2_alkafri
- sagittal_spider

## Artefactos finales

- `{FINAL_MODELS_ROOT / "axial_t2_alkafri_final_best.pt"}`
- `{FINAL_MODELS_ROOT / "sagittal_spider_multiclass_final_best.pt"}`
- `config/model_registry_final.json`

## Métricas principales

### Axial T2 Al-Kafri/Sudirman

- Test Dice útil excluyendo raw_0: 0.8167
- Test Dice macro no-bg incluyendo raw_0: 0.6587
- Nota: raw_0 se reporta por separado porque E11 lo documentó como clase minoritaria/intermitente.

### Sagital SPIDER

- Holdout/Test Dice macro no-bg: 0.8311
- Val Dice macro no-bg: 0.8392

## Checks

- all_checks_ok: {all_ok}
- modelos finales con hash SHA256: {bool(artifacts_df["sha256"].notna().all())}
- métricas limitadas al alcance final: true

## Próximo bloque

PF-008 a PF-011: convertir inferencia E13 a módulos productivos del servicio Python.
"""

closure_path = REPO_ROOT / "backlogProducto" / "PF004_PF007_resultados_cierre.md"
closure_path.write_text(closure_md, encoding="utf-8")

print("Report:", report_path)
print(json.dumps(report, indent=2, ensure_ascii=False))
print("Closure draft:", closure_path)


Report: /content/drive/MyDrive/PFI_MVP/results/PF004_PF007_final_models/PF004_PF007_report.json
{
  "notebook": "29_PF004_PF007_final_models_registry_metrics",
  "goal": "consolidate final model artifacts, final model registry and final metrics",
  "timestamp": "2026-06-30T22:00:40",
  "inputs": {
    "data_freeze_config": "/content/drive/MyDrive/PFI_MVP/repo/config/data_freeze_config.json",
    "axial_source_checkpoint": "/content/drive/MyDrive/PFI_MVP/models/E10_axial_t2_final_training_clean_best.pt",
    "sagittal_source_checkpoint": "/content/drive/MyDrive/PFI_MVP/models/E12_sagittal_multiclass_final_best.pt"
  },
  "outputs": {
    "final_models_root": "/content/drive/MyDrive/PFI_MVP/models/final",
    "model_artifacts_csv": "/content/drive/MyDrive/PFI_MVP/results/PF004_PF007_final_models/PF004_PF007_model_artifacts.csv",
    "model_registry_results": "/content/drive/MyDrive/PFI_MVP/results/PF004_PF007_final_models/PF006_model_registry_final.json",
    "model_registry_repo": "/con

## 8. Commit sugerido desde Colab

```bash
%cd /content/drive/MyDrive/PFI_MVP/repo
!git status
!git add config/model_registry_final.json docs/PF004_PF007_final_models_metrics.md backlogProducto/PF004_PF007_resultados_cierre.md notebooks/29_PF004_PF007_final_models_registry_metrics.ipynb
!git commit -m "Close PF004 PF007 final models registry and metrics"
!git push
```

Los checkpoints `.pt` quedan en Drive (`/models/final/`) y normalmente no se suben al repo por tamaño.
